In [ ]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import shap

from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
# Load Processed Datasets

X_train = pd.read_csv(
    "../data/processed/X_train.csv"
)

X_test = pd.read_csv(
    "../data/processed/X_test.csv"
)

y_train = pd.read_csv(
    "../data/processed/y_train.csv"
)

y_test = pd.read_csv(
    "../data/processed/y_test.csv"
)

In [ ]:
# Dataset Verification

print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Target :", y_train.shape)

print("Testing Target :", y_test.shape)

In [ ]:
# Remove Identifier Columns

identifier_columns = [

    "Run",

    "Event"

]

X_train = X_train.drop(

    columns=identifier_columns,

    errors="ignore"

)

X_test = X_test.drop(

    columns=identifier_columns,

    errors="ignore"

)

print("Identifier Columns Removed")

In [ ]:
# Convert Target Variables to Series

y_train = y_train.squeeze()

y_test = y_test.squeeze()

In [ ]:
# Load Best Model

best_model = joblib.load(

    "../models/best_model.pkl"

)

print("Best Model Loaded Successfully")

In [ ]:
# Calculate Feature Importance

feature_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": best_model.feature_importances_

})

feature_importance = feature_importance.sort_values(

    by="Importance",

    ascending=False

).reset_index(

    drop=True

)

display(feature_importance)

In [ ]:
# Visualize Feature Importance

plt.figure(figsize=(10, 8))

sns.barplot(

    data=feature_importance,

    x="Importance",

    y="Feature",

    color="steelblue"

)

plt.title("CatBoost Feature Importance")

plt.xlabel("Importance Score")

plt.ylabel("Feature")

plt.tight_layout()

plt.show()

In [ ]:
# Display Most Important Sensors

display(

    feature_importance.head(10)

)

In [ ]:
# Save Feature Importance

os.makedirs(

    "../results",

    exist_ok=True

)

feature_importance.to_csv(

    "../results/feature_importance.csv",

    index=False

)

print("Feature Importance Saved Successfully")

In [ ]:
# Calculate Permutation Importance

permutation = permutation_importance(

    estimator=best_model,

    X=X_test,

    y=y_test,

    scoring="neg_root_mean_squared_error",

    n_repeats=10,

    random_state=42,

    n_jobs=-1

)

In [ ]:
# Create Permutation Importance Table

permutation_importance_df = pd.DataFrame({

    "Feature": X_test.columns,

    "Importance": permutation.importances_mean,

    "Standard Deviation": permutation.importances_std

})

permutation_importance_df = permutation_importance_df.sort_values(

    by="Importance",

    ascending=False

).reset_index(

    drop=True

)

display(permutation_importance_df)

In [ ]:
# Visualize Permutation Importance

plt.figure(figsize=(10, 8))

sns.barplot(

    data=permutation_importance_df,

    x="Importance",

    y="Feature",

    color="darkorange"

)

plt.title("Permutation Feature Importance")

plt.xlabel("Importance Score")

plt.ylabel("Feature")

plt.tight_layout()

plt.show()

In [ ]:
# Display Most Important Sensors

display(

    permutation_importance_df.head(10)

)

In [ ]:
# Save Permutation Importance

permutation_importance_df.to_csv(

    "../results/permutation_importance.csv",

    index=False

)

print("Permutation Importance Saved Successfully")

In [ ]:
# Create a Sample for SHAP Analysis

X_shap = X_test.sample(

    n=1000,

    random_state=42

)

In [ ]:
# Initialize SHAP Explainer

explainer = shap.TreeExplainer(

    best_model

)

In [ ]:
# Calculate SHAP Values

shap_values = explainer.shap_values(

    X_shap

)

In [ ]:
# Visualize SHAP Summary Plot

shap.summary_plot(

    shap_values,

    X_shap,

    show=False

)

plt.tight_layout()

plt.show()

In [ ]:
# Visualize SHAP Feature Importance

shap.summary_plot(

    shap_values,

    X_shap,

    plot_type="bar",

    show=False

)

plt.tight_layout()

plt.show()

In [ ]:
# Create SHAP Importance Table

shap_importance = pd.DataFrame({

    "Feature": X_shap.columns,

    "Importance": np.abs(

        shap_values

    ).mean(

        axis=0

    )

})

shap_importance = shap_importance.sort_values(

    by="Importance",

    ascending=False

).reset_index(

    drop=True

)

display(shap_importance)

In [ ]:
# Save SHAP Importance

shap_importance.to_csv(

    "../results/shap_importance.csv",

    index=False

)

print("SHAP Importance Saved Successfully")

In [ ]:
# Combine Feature Importance Scores

consensus_importance = feature_importance.rename(

    columns={

        "Importance": "CatBoost"

    }

).merge(

    permutation_importance_df[

        [

            "Feature",

            "Importance"

        ]

    ].rename(

        columns={

            "Importance": "Permutation"

        }

    ),

    on="Feature"

).merge(

    shap_importance.rename(

        columns={

            "Importance": "SHAP"

        }

    ),

    on="Feature"

)

In [ ]:
# Normalize Importance Scores

importance_columns = [

    "CatBoost",

    "Permutation",

    "SHAP"

]

for column in importance_columns:

    consensus_importance[column] = (

        consensus_importance[column]

        /

        consensus_importance[column].max()

    )

In [ ]:
# Calculate Consensus Importance Score

consensus_importance["Consensus"] = (

    consensus_importance[

        importance_columns

    ].mean(

        axis=1

    )

)

In [ ]:
# Rank Sensors

consensus_importance = consensus_importance.sort_values(

    by="Consensus",

    ascending=False

).reset_index(

    drop=True

)

display(consensus_importance)

In [ ]:
# Display Most Important Sensors

display(

    consensus_importance.head(10)

)

In [ ]:
# Visualize Consensus Feature Importance

plt.figure(figsize=(10,8))

sns.barplot(

    data=consensus_importance,

    x="Consensus",

    y="Feature",

    color="forestgreen"

)

plt.title("Consensus Feature Importance")

plt.xlabel("Consensus Importance Score")

plt.ylabel("Feature")

plt.tight_layout()

plt.show()

In [ ]:
# Save Consensus Feature Importance

consensus_importance.to_csv(

    "../results/consensus_feature_importance.csv",

    index=False

)

print("Consensus Feature Importance Saved Successfully")

In [ ]:
# Define Sensor Subsets

feature_counts = [

    len(consensus_importance),

    15,

    12,

    10,

    8,

    5

]

feature_counts = [

    count

    for count in feature_counts

    if count <= len(consensus_importance)

]

In [ ]:
# Evaluate Different Sensor Sets

sensor_results = []

for count in feature_counts:

    selected_features = consensus_importance[

        "Feature"

    ].head(

        count

    ).tolist()

    model = best_model.__class__(

        random_state=42,

        verbose=0

    )

    model.fit(

        X_train[selected_features],

        y_train

    )

    predictions = model.predict(

        X_test[selected_features]

    )

    mae = mean_absolute_error(

        y_test,

        predictions

    )

    rmse = np.sqrt(

        mean_squared_error(

            y_test,

            predictions

        )

    )

    r2 = r2_score(

        y_test,

        predictions

    )

    sensor_results.append({

        "Sensors": count,

        "MAE": mae,

        "RMSE": rmse,

        "R²": r2

    })

print("Sensor Purging Completed")

In [ ]:
# Create Sensor Purging Results

sensor_results_df = pd.DataFrame(

    sensor_results

)

display(sensor_results_df)

In [ ]:
# Visualize Sensor Trade-off

plt.figure(figsize=(8,6))

plt.plot(

    sensor_results_df["Sensors"],

    sensor_results_df["RMSE"],

    marker="o"

)

plt.gca().invert_xaxis()

plt.title(

    "Sensor Trade-off Curve"

)

plt.xlabel(

    "Number of Sensors"

)

plt.ylabel(

    "RMSE"

)

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
# Recommend Optimal Sensor Set

baseline_rmse = sensor_results_df.iloc[0]["RMSE"]

sensor_results_df["RMSE Increase (%)"] = (

    (

        sensor_results_df["RMSE"]

        - baseline_rmse

    )

    /

    baseline_rmse

) * 100

display(sensor_results_df)

In [ ]:
# Save Sensor Purging Results

sensor_results_df.to_csv(

    "../results/sensor_purging_results.csv",

    index=False

)

print("Sensor Purging Results Saved Successfully")

In [ ]:
# Save Recommended Sensors

selected_features = consensus_importance[

    "Feature"

].head(

    10

)

selected_features.to_csv(

    "../results/selected_features.csv",

    index=False

)

print("Selected Sensors Saved Successfully")